## Import dependencies

In [88]:
!python -m pip install --quiet -r ../requirements.txt


[notice] A new release of pip is available: 23.2.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [89]:
import pandas as pd

df = pd.read_csv("../data/processed/telco-churn-cleaned.csv")

df.shape

(7043, 21)

### 1. Tạo X, y

In [90]:
X = df.drop(columns = ['customerID', 'Churn'])
y = df['Churn']

In [91]:
print(X.shape)
print(y.shape)

(7043, 19)
(7043,)


### 2. Train/test split

Vì target imbalance nên phải dùng stratify

In [92]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y, 
    test_size=0.2,
    random_state=6769,
    stratify=y,
)

In [93]:
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (5634, 19)
X_test: (1409, 19)
y_train: (5634,)
y_test: (1409,)


In [94]:
print("Full:", y.value_counts(normalize=True))
print("Train:", y_train.value_counts(normalize=True))
print("Test:", y_test.value_counts(normalize=True))

Full: Churn
No     0.73463
Yes    0.26537
Name: proportion, dtype: float64
Train: Churn
No     0.734647
Yes    0.265353
Name: proportion, dtype: float64
Test: Churn
No     0.734564
Yes    0.265436
Name: proportion, dtype: float64


### Lưu ý quan trọng: Phải split trước khi preprocessing

### 3. Encoding

In [95]:
X_train.dtypes

gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges        float64
dtype: object

#### Khai báo các cột

In [96]:
numerical_features = [
    "tenure",
    "MonthlyCharges",
    "TotalCharges"
]

In [97]:
binary_features = [
    "SeniorCitizen",
    "Partner",
    "Dependents",
    "PhoneService",
    "PaperlessBilling"
]

In [98]:
multiclass_features = [
    "gender",
    "MultipleLines",
    "InternetService",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
    "Contract",
    "PaymentMethod"
]

In [99]:
len(numerical_features)

3

In [100]:
len(binary_features)

5

In [101]:
len(multiclass_features)

11

In [102]:
X_train_pre = X_train.copy()
X_test_pre = X_test.copy()

#### Xử lý binary

In [103]:
yes_no_mapping = {
    "No": 0,
    "Yes": 1,
}

In [104]:
for col in binary_features:
    if col != "SeniorCitizen":
        X_train_pre[col] = X_train_pre[col].map(yes_no_mapping)
        X_test_pre[col] = X_test_pre[col].map(yes_no_mapping)

In [105]:
for col in binary_features:
    print(
        col,
        X_train_pre[col].unique()
    )

SeniorCitizen [0 1]
Partner [0 1]
Dependents [0 1]
PhoneService [1 0]
PaperlessBilling [0 1]


#### Khai báo và xử lý OneHotEncode

In [106]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(
    handle_unknown='ignore',
    sparse_output=False,
)

In [107]:
encoder.fit(X_train_pre[multiclass_features])

,"sparse_output sparse_output: bool, default=TrueWhen ``True``, it returns a SciPy sparse matrix/arrayin ""Compressed Sparse Row"" (CSR) format... versionadded:: 1.2 `sparse` was renamed to `sparse_output`",False
,"handle_unknown handle_unknown: {'error', 'ignore', 'infrequent_if_exist', 'warn'}, default='error'Specifies the way unknown categories are handled during :meth:`transform`.- 'error' : Raise an error if an unknown category is present during transform.- 'ignore' : When an unknown category is encountered during transform, the resulting one-hot encoded columns for this feature will be all zeros. In the inverse transform, an unknown category will be denoted as None.- 'infrequent_if_exist' : When an unknown category is encountered during transform, the resulting one-hot encoded columns for this feature will map to the infrequent category if it exists. The infrequent category will be mapped to the last position in the encoding. During inverse transform, an unknown category will be mapped to the category denoted `'infrequent'` if it exists. If the `'infrequent'` category does not exist, then :meth:`transform` and :meth:`inverse_transform` will handle an unknown category as with `handle_unknown='ignore'`. Infrequent categories exist based on `min_frequency` and `max_categories`. Read more in the :ref:`User Guide <encoder_infrequent_categories>`.- 'warn' : When an unknown category is encountered during transform a warning is issued, and the encoding then proceeds as described for `handle_unknown=""infrequent_if_exist""`... versionchanged:: 1.1 `'infrequent_if_exist'` was added to automatically handle unknown categories and infrequent categories... versionadded:: 1.6 The option `""warn""` was added in 1.6.",'ignore'
,"categories categories: 'auto' or a list of array-like, default='auto'Categories (unique values) per feature:- 'auto' : Determine categories automatically from the training data.- list : ``categories[i]`` holds the categories expected in the ith column. The passed categories should not mix strings and numeric values within a single feature, and should be sorted in case of numeric values.The used categories can be found in the ``categories_`` attribute... versionadded:: 0.20",'auto'
,"drop drop: {'first', 'if_binary'} or an array-like of shape (n_features,), default=NoneSpecifies a methodology to use to drop one of the categories perfeature. This is useful in situations where perfectly collinearfeatures cause problems, such as when feeding the resulting datainto an unregularized linear regression model.However, dropping one category breaks the symmetry of the originalrepresentation and can therefore induce a bias in downstream models,for instance for penalized linear classification or regression models.- None : retain all features (the default).- 'first' : drop the first category in each feature. If only one category is present, the feature will be dropped entirely.- 'if_binary' : drop the first category in each feature with two categories. Features with 1 or more than 2 categories are left intact.- array : ``drop[i]`` is the category in feature ``X[:, i]`` that should be dropped.When `max_categories` or `min_frequency` is configured to groupinfrequent categories, the dropping behavior is handled after thegrouping... versionadded:: 0.21 The parameter `drop` was added in 0.21... versionchanged:: 0.23 The option `drop='if_binary'` was added in 0.23... versionchanged:: 1.1 Support for dropping infrequent categories.",None
,"dtype dtype: number type, default=np.float64Desired dtype of output.",<class 'numpy.float64'>
,"min_frequency min_frequency: int or float, default=NoneSpecifies the minimum frequency below which a category will beconsidered infrequent.- If `int`, categories with a smaller cardinality will be considered infrequent.- If `float`, categories with a smaller cardinality than `min_frequency * n_samples` will be considered infrequent... versionadded:: 1.1 Read more in the :ref:`User Guide <encoder_infreque

In [108]:
X_train_multi = encoder.transform(
    X_train_pre[multiclass_features]
)

In [109]:
X_test_multi = encoder.transform(
    X_test_pre[multiclass_features]
)

In [110]:
print(X_train_multi.shape)
print(X_test_multi.shape)

(5634, 33)
(1409, 33)


#### Kiểm tra Encoder đã học được gì

In [112]:
encoded_feature_names = encoder.get_feature_names_out(
    multiclass_features
)
encoded_feature_names

array(['gender_Female', 'gender_Male', 'MultipleLines_No',
       'MultipleLines_No phone service', 'MultipleLines_Yes',
       'InternetService_DSL', 'InternetService_Fiber optic',
       'InternetService_No', 'OnlineSecurity_No',
       'OnlineSecurity_No internet service', 'OnlineSecurity_Yes',
       'OnlineBackup_No', 'OnlineBackup_No internet service',
       'OnlineBackup_Yes', 'DeviceProtection_No',
       'DeviceProtection_No internet service', 'DeviceProtection_Yes',
       'TechSupport_No', 'TechSupport_No internet service',
       'TechSupport_Yes', 'StreamingTV_No',
       'StreamingTV_No internet service', 'StreamingTV_Yes',
       'StreamingMovies_No', 'StreamingMovies_No internet service',
       'StreamingMovies_Yes', 'Contract_Month-to-month',
       'Contract_One year', 'Contract_Two year',
       'PaymentMethod_Bank transfer (automatic)',
       'PaymentMethod_Credit card (automatic)',
       'PaymentMethod_Electronic check', 'PaymentMethod_Mailed check'],
      dty

#### Chuyển đổi lại thành DataFrame

In [114]:
X_train_multi = pd.DataFrame(
    X_train_multi,
    columns = encoded_feature_names,
    index = X_train_pre.index
)

X_test_multi = pd.DataFrame(
    X_test_multi,
    columns=encoded_feature_names,
    index=X_test_pre.index
)

#### Scale lại numerical number

In [126]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

scaler.fit(
    X_train_pre[numerical_features]
)

StandardScaler()

In [127]:
X_train_num = scaler.transform(
    X_train_pre[numerical_features]
)
X_test_num = scaler.transform(
    X_test_pre[numerical_features]
)

In [128]:
print("Mean:", scaler.mean_)
print("Scale:", scaler.scale_)

Mean: [  32.54543841   64.54108981 2286.5335197 ]
Scale: [  24.60193153   30.18481355 2275.86007194]


In [129]:
print(X_train_num.mean(axis=0))
print(X_test_num.std(axis=0))

[1.09091137e-16 1.75933105e-16 2.58854981e-16]
[0.99048191 0.98330561 0.9794766 ]


#### Chuyển lại thành DataFrame

In [130]:
X_train_num = pd.DataFrame(
    X_train_num,
    columns=numerical_features,
    index=X_train_pre.index
)
X_test_num = pd.DataFrame(
    X_test_num,
    columns=numerical_features,
    index=X_test_pre.index
)

#### Lấy Binary

In [131]:
X_train_binary = X_train_pre[binary_features].copy()
X_test_binary = X_test_pre[binary_features].copy()

#### Ghép lại tất cả

In [132]:
X_train_ready = pd.concat(
    [
        X_train_num,
        X_train_binary,
        X_train_multi,
    ],
    axis=1
)

X_test_ready = pd.concat(
    [
        X_test_num,
        X_test_binary,
        X_test_multi,
    ],
    axis=1
)

#### Validation

In [133]:
print(X_train_ready.shape)
print(X_test_ready.shape)

(5634, 41)
(1409, 41)


Tất cả loại dữ liệu khác đã được preprocessing và ghép lại

In [134]:
assert list(X_train_ready.columns) == list(
    X_test_ready.columns
)

assert X_train_ready.isna().sum().sum() == 0

assert X_test_ready.isna().sum().sum() == 0

In [135]:
print(
    X_train_ready.select_dtypes(
        include=["object"]
    ).columns
)

Index([], dtype='str')


In [136]:
X_train_ready[
    numerical_features
].describe()

,tenure,MonthlyCharges,TotalCharges
count,5.634000e+03,5.634000e+03,5.634000e+03
mean,1.090911e-16,1.759331e-16,2.588550e-16
std,1.000089e+00,1.000089e+00,1.000089e+00
min,-1.322881e+00,-1.528619e+00,-1.004690e+00
25%,-9.570565e-01,-9.852998e-01,-8.277622e-01
50%,-1.441122e-01,1.874754e-01,-3.938329e-01
75%,9.533626e-01,8.347380e-01,6.684523e-01
max,1.603718e+00,1.795900e+00,2.811362e+00


Tất cả validation đã được pass. Giờ chúng ta đã có `model-ready feature matrix`